<a href="https://colab.research.google.com/github/mayurpophale/data_science/blob/main/CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.preprocessing import LabelEncoder,StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Perceptron
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Conv2D,Flatten,MaxPooling2D,Dropout
from tensorflow.keras.utils import to_categorical

In [3]:
df = pd.read_csv('/content/mnist_train.csv')
df_test = pd.read_csv('/content/mnist_test.csv')

UnicodeDecodeError: 'utf-8' codec can't decode byte 0x9d in position 10: invalid start byte

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
X_train = df.drop('label',axis=1).values
y_train = df['label'].values
X_test = df_test.drop('label',axis=1).values
y_test = df_test['label'].values

In [ ]:
X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

In [ ]:
X_train_img = X_train.reshape(-1,28,28)
X_test_img = X_test.reshape(-1,28,28)

In [ ]:
y_train_cat = to_categorical(y_train,10)
y_test_cat = to_categorical(y_test,10)

## **Percetron**

In [ ]:
percetron = Sequential([
    Flatten(input_shape=(28,28)),
    Dense(10,activation='softmax')]
)

In [ ]:
percetron.compile(optimizer='sgd',loss='categorical_crossentropy',metrics=['accuracy'])

In [ ]:
hist_percetron = percetron.fit(X_train_img,y_train_cat,epochs=5,batch_size=32,
                               validation_data=(X_test_img,y_test_cat),verbose=1)

In [ ]:
acc_per = percetron.evaluate(X_test_img,y_test_cat)[1]

## **ANN**

In [ ]:
ann = Sequential([
    Flatten(input_shape=(28,28)),
    Dense(128,activation='relu'),
    Dense(64,activation='relu'),
    Dense(10,activation='softmax')]
)

In [ ]:
ann.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])

In [ ]:
hist_ann = ann.fit(X_train_img,y_train_cat,epochs=5,batch_size=32,
                   validation_data=(X_test_img,y_test_cat),verbose=1)

In [ ]:
acc_ann = ann.evaluate(X_test_img,y_test_cat)[1]

## **CNN**

In [ ]:
X_test_img_cnn = X_test_img.reshape(-1,28,28,1)
X_train_img_cnn = X_train_img.reshape(-1,28,28,1)

In [ ]:
cnn = Sequential([
    Conv2D(32,(3,3),activation='relu',input_shape=(28,28,1)),
    MaxPooling2D((2,2)),
    Conv2D(64,(3,3),activation='relu'),
    MaxPooling2D((2,2)),
    Flatten(),
    Dense(128,activation='relu'),
    Dropout(0.5),
    Dense(10,activation='softmax')]
)

In [ ]:
cnn.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])

In [ ]:
hist_cnn = cnn.fit(X_train_img_cnn,y_train_cat,epochs=5,batch_size=32,
                   validation_data=(X_test_img_cnn,y_test_cat),verbose=1)

In [ ]:
acc_cnn = cnn.evaluate(X_test_img_cnn,y_test_cat)[1]

## **Comparision**

In [ ]:
def plot_training(history, title):
    plt.figure(figsize=(12,4))
    plt.subplot(1,2,1)
    plt.plot(history.history['accuracy'], label="Train")
    plt.plot(history.history['val_accuracy'], label="Val")
    plt.title(f"{title} Accuracy")
    plt.legend()

    plt.subplot(1,2,2)
    plt.plot(history.history['loss'], label="Train")
    plt.plot(history.history['val_loss'], label="Val")
    plt.title(f"{title} Loss")
    plt.legend()
    plt.show()

In [ ]:
plot_training(hist_percetron, "Perceptron")


In [ ]:
plot_training(hist_ann, "ANN")

In [ ]:
plot_training(hist_cnn, "CNN")

In [ ]:
plt.figure(figsize=(10,6))
plt.plot(hist_percetron.history['val_accuracy'], label="Perceptron")
plt.plot(hist_ann.history['val_accuracy'], label="ANN")
plt.plot(hist_cnn.history['val_accuracy'], label="CNN")
plt.title("Validation Accuracy Comparison")
plt.xlabel("Epochs")
plt.ylabel("Val Accuracy")
plt.legend()
plt.show()

In [ ]:
def show_side_by_side(models, model_names, X, X_cnn, y_true, n=5):
    idxs = np.random.choice(len(X), n, replace=False)
    plt.figure(figsize=(15, 6))
    for i, idx in enumerate(idxs):
        plt.subplot(2, n, i+1)
        plt.imshow(X[idx].reshape(28, 28), cmap="gray")
        plt.axis("off")
        plt.title(f"True: {y_true[idx]}")
        preds = [np.argmax(model.predict(X_cnn[idx].reshape(1, 28, 28, 1) if name == "CNN" else X[idx].reshape(1, 28, 28)))
                 for model, name in zip(models, model_names)]
        plt.subplot(2, n, n+i+1)
        plt.axis("off")
        plt.title("\n".join(f"{n}: {p}" for n, p in zip(model_names, preds)))
    plt.tight_layout()
    plt.show()

In [ ]:
show_side_by_side([percetron, ann, cnn], ["Perceptron", "ANN", "CNN"], X_test, X_test_img_cnn, y_test, n=8)

In [ ]:
#confusion matrix
y_pred = cnn.predict(X_test_img_cnn)
y_pred = np.argmax(y_pred, axis=1)
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=range(10), yticklabels=range(10))
plt.xlabel('Predicted')
plt.ylabel('True')

In [ ]:
#Final Acc Comparision
print(f"Perceptron Accuracy: {acc_per:.4f}")
print(f"ANN Accuracy: {acc_ann:.4f}")
print(f"CNN Accuracy: {acc_cnn:.4f}")
plt.figure(figsize=(10,6))
plt.bar(['Perceptron', 'ANN', 'CNN'], [acc_per, acc_ann, acc_cnn])
plt.ylabel('Accuracy')
plt.ylim([0, 1])
plt.title('Final Accuracy Comparison')
plt.show()